# 8주차 중간고사: 고객 리뷰 데이터 분석 따라하기
학번: ______  성명: ______

**실행 순서:** 파일→Drive에 사본 저장 → 런타임→모두 실행. C1~C6(총 40점)은 예제를 실행한 뒤 자신의 결과 해석과 개선을 추가하세요. 코드는 기본 동작 예시이며 학생의 분석·판단이 평가 대상입니다.

별도 API 키 불필요. 분석 대상은 시험용 합성 데이터입니다.

## 수행평가 문제 지시문 (첫 번째로 읽을 것)
**역할:** 여러분은 고객 만족도 분석 담당자입니다. 제공된 시험용 CSV를 Colab에서 읽어 데이터 품질을 점검하고, 서비스별 만족도를 비교하며, 월별 추이와 감성분석을 설명하는 보고서를 완성하십시오.
**반드시 완성할 결과물:** C1 원본 구조 표 / C2 정제 전후 건수 / C3 서비스별 통계표 / C4 월별 선그래프와 서비스별 막대그래프 / C5 감성 교차표 및 일치율 / C6 단어 상위 10개 표와 그래프 / D 해석 답변 5개.
### 완성된 결과의 예상 형태 — 실제 정답 수치는 직접 산출
| 단계 | 예상 출력 화면 | 학생이 작성할 내용 |
|---|---|---|
| C1 | (행 수, 열 수), 5개 필드의 자료형과 5행 미리보기 | 각 열 의미 |
| C2 | 결측·중복 건수와 정제 전후 표 | 제거 이유 |
| C3 | category / review_count / mean_rating 표 | 가장 높은 분야 |
| C4 | X=월, Y=별점 선그래프; X=서비스, Y=평균별점 막대그래프 | 추세 해석 |
| C5 | 실제 별점 감성과 규칙 감성 교차표, 일치율 | 직접 고친 키워드와 불일치 이유 |
| C6 | 긍정·부정 상위 단어 10개 표, 빈도 막대그래프 | 단어 해석과 한계 |
**실행 방법:** ① 파일 > Drive에 사본 저장 ② 위에서부터 Shift+Enter로 셀 실행 ③ 결과 확인 ④ 학생 답변 및 키워드 수정 ⑤ 런타임 > 모두 실행 ⑥ ipynb, 답안 PDF, 생성 CSV, 그림 PNG 제출.
**주의:** 예상 화면에는 실제 평점이나 감성 분류 정답을 공개하지 않습니다. 숫자와 해석은 직접 구합니다.
[시험지 및 채점항목](https://github.com/itlects/2026-2_ai-data/blob/main/Week08_Midterm/README.md)

## 0. 실습 준비
CSV는 GitHub 공개 경로에서 읽습니다. 읽기에 실패하면 직접 다운로드한 파일을 왼쪽 파일 메뉴에 업로드해 `pd.read_csv('midterm_reviews.csv')`로 변경합니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re
from collections import Counter
from IPython.display import display

DATA_URL = 'https://raw.githubusercontent.com/itlects/2026-2_ai-data/main/Week08_Midterm/data/midterm_reviews.csv'
plt.rcParams['axes.unicode_minus'] = False
print('환경 준비 완료')

## C1. 데이터 읽기와 구조 확인 (5점)

**예상 출력: 원본 크기, 5개 열의 자료형, 첫 5행이 표로 나타납니다.**
`read_csv`는 CSV를 표로 읽습니다. 아래 셀을 실행해 행·열 수, 자료형, 상위 5개 행을 확인하세요. **작성:** 자료의 열이 무엇을 뜻하는지 한 문장으로 적으세요.

In [ ]:
df = pd.read_csv(DATA_URL, dtype={'review_id':'string'})
print('행/열:',df.shape)
print('자료형:')
print(df.dtypes)
display(df.head(5))
# 학생 답변: 이 데이터의 열은 ...

## C2. 데이터 정제 (5점)

**예상 출력: 열별 결측 수·ID 중복 건수·정제 전후 건수가 나옵니다.**
1) 비어 있는 리뷰와 공백만 있는 리뷰를 결측으로 통일합니다. 2) 같은 `review_id`가 두 번 나오면 첫 기록만 남깁니다. 3) 날짜를 날짜형으로 변환합니다. **작성:** 각 단계에서 제거한 건수를 적으세요.

In [ ]:
before = len(df)
work = df.copy()
work['review_text'] = work['review_text'].astype('string').str.strip()
work['review_text'] = work['review_text'].replace('', pd.NA)
print('결측 건수:\n',work.isna().sum())
print('review_id 중복 건수:',work.duplicated(subset=['review_id']).sum())
work = work.dropna(subset=['review_text']).drop_duplicates(subset=['review_id']).copy()
work['date'] = pd.to_datetime(work['date'],errors='coerce')
work = work.dropna(subset=['date'])
clean = work.sort_values('date').reset_index(drop=True)
print('정제 전:', before, '정제 후:',len(clean), '제거:',before-len(clean))
display(clean.head())
# 학생 답변: 무엇이 왜 제거되었나요?

## C3. 서비스별 평균 별점 (5점)

**예상 출력: category, review_count, mean_rating 열이 있는 서비스별 요약표입니다.**
`groupby('category')`로 범주별 건수와 평균을 구하고 CSV로 저장합니다. **작성:** 가장 높은 서비스와 표본 건수를 적으세요.

In [ ]:
category_stats = clean.groupby('category').agg(review_count=('rating','size'),mean_rating=('rating','mean')).reset_index()
display(category_stats)
category_stats.to_csv('category_summary.csv',index=False,encoding='utf-8-sig')
# 학생 답변: 가장 높은 서비스는 ... 이유는 데이터만으로 알 수 있는가?

## C4. 월별·범주별 결과 시각화 (10점)

**예상 출력: 월별 평균 별점 선그래프와 분야별 평균 별점 막대그래프를 각각 확인합니다.**
월별 평점은 선그래프, 서비스별 평균 평점은 막대그래프로 제시합니다. **작성:** x축·y축의 뜻을 확인하고 가장 높고 낮은 월을 적으세요.

In [ ]:
clean['month'] = clean['date'].dt.to_period('M').astype(str)
monthly = clean.groupby('month')['rating'].mean()
plt.figure(figsize=(9,4))
monthly.plot(marker='o')
plt.title('Monthly average rating');plt.xlabel('Month');plt.ylabel('Average rating');plt.xticks(rotation=45);plt.tight_layout()
plt.savefig('monthly_ratings.png',dpi=140);plt.show()
plt.figure(figsize=(7,4))
plt.bar(category_stats['category'],category_stats['mean_rating'])
plt.title('Average rating by service');plt.xlabel('Service');plt.ylabel('Average rating');plt.ylim(0,5);plt.tight_layout()
plt.savefig('category_ratings.png',dpi=140);plt.show()
# 학생 답변: 월별 그래프의 변화는 ...

## C5. 규칙 기반 감성분류 (10점)

**예상 출력: 실제 별점 감성×규칙 감성 교차표, 제외 건수, 일치율을 확인합니다.**
아래 긍정·부정 키워드는 **예시**입니다. 최소 2개 이상의 표현을 직접 추가하거나 수정하고, 선택한 이유를 작성하세요. 긍정과 부정이 함께 나오면 점수가 같을 경우 미판정입니다. 별점 3점은 중립으로 두며 **중립·미판정은 일치율 분모에서 제외**합니다.

In [ ]:
POS_WORDS = ['좋','만족','친절','훌륭','추천','맛있','빠르']
NEG_WORDS = ['나쁘','아쉽','불편','늦','비싸','부족','없']
# TODO: 위 목록에 각자 2개 이상 추가하거나 변경하고 근거 작성

def classify(text):
    s=str(text)
    p=sum(w in s for w in POS_WORDS)
    n=sum(w in s for w in NEG_WORDS)
    return '긍정' if p>n else ('부정' if n>p else '미판정')

clean['rule_sentiment'] = clean['review_text'].map(classify)
clean['rating_sentiment'] = clean['rating'].map(lambda x:'긍정' if x>=4 else ('부정' if x<=2 else '중립'))
ct=pd.crosstab(clean['rating_sentiment'],clean['rule_sentiment'])
display(ct)
valid=clean[(clean['rating_sentiment']!='중립')&(clean['rule_sentiment']!='미판정')]
rate=(valid['rating_sentiment']==valid['rule_sentiment']).mean() if len(valid) else float('nan')
print('평가 대상:',len(valid),'제외:',len(clean)-len(valid),'일치율:',rate)
ct.to_csv('sentiment_crosstab.csv',encoding='utf-8-sig')
# 학생 답변: 어떤 키워드를 변경했고 왜 그랬나요?

## C6. 단어 빈도 분석 (5점)

**예상 출력: 전체 단어 TOP10 그래프 및 긍정/부정 그룹의 단어 빈도표가 나옵니다.**
초급 실습에서는 공백으로 어절을 구분하고, 특수문자를 제거합니다. 분석 기준을 작성하고 결과의 한계를 제시하세요. 한국어의 정확한 형태소 분석은 아닙니다.

In [ ]:
STOPWORDS={'그리고','정말','너무','하지만','그냥','조금','아주'}
def tokenize(s):
    return [w for w in re.sub(r'[^가-힣a-zA-Z0-9\s]',' ',str(s)).split() if len(w)>1 and w not in STOPWORDS]

all_words=[w for s in clean['review_text'] for w in tokenize(s)]
top10=pd.DataFrame(Counter(all_words).most_common(10),columns=['word','count'])
display(top10)
plt.figure(figsize=(9,4));plt.bar(range(len(top10)),top10['count']);plt.xticks(range(len(top10)),top10['word'],rotation=35);plt.title('Top 10 words');plt.xlabel('Word');plt.ylabel('Frequency');plt.tight_layout();plt.savefig('top10_words.png',dpi=140);plt.show()
top10.to_csv('top10_words.csv',index=False,encoding='utf-8-sig')

# 감성별 상위 10개 단어를 각각 집계 (미판정은 제외)
for sentiment in ['긍정','부정']:
    subset=clean.loc[clean['rule_sentiment']==sentiment,'review_text']
    words=[w for line in subset for w in tokenize(line)]
    freq=pd.DataFrame(Counter(words).most_common(10),columns=['word','count'])
    print(sentiment, '리뷰 수:',len(subset));display(freq)
    freq.to_csv('top10_'+('positive' if sentiment=='긍정' else 'negative')+'.csv',index=False,encoding='utf-8-sig')
# 학생 답변: 두 그룹의 주요 단어가 어떻게 다른가요?

## D. 실행 결과 질문 (20점: 5문항 × 4점)
15. 정제 전후 리뷰 건수와 제거 이유 (4점)
16. 평균 별점이 가장 높은 서비스, 평점·표본 수·해석상 주의점 (4점)
17. 최고·최저 월과 평균값 및 월별 추세 설명 (4점)
18. 규칙 감성과 별점이 불일치한 리뷰 ID·원문과 그 이유 (4점)
19. 긍정·부정 빈도 단어 차이, AI/본인 해석문에서 사실과 추정 구분 (4점)

**제출:** 실행된 ipynb, 답안 PDF, CSV 3개, 그래프 PNG 3개, AI 활용 기록. 파일명에 학번·성명을 포함하세요.